**Script to estimate population exposed to high heat, based on yearly CDDs**

In [1]:
# import packages 
import os
import xarray as xr
import netCDF4
import numpy as np
import rioxarray as rxr
import pandas as pd
import rasterio
from rasterio.crs import CRS
from rasterio.enums import Resampling

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
import matplotlib.colors as colors

import pym

**READ IN DATA**

In [3]:
# read netcdf data using xarray
def read_netcdf(filepath):
    dataset = xr.open_dataset(filepath)
    return dataset

ROOT_DIR = r'X:\user\crassierc\code\cooling\CoolingGap'
os.chdir(ROOT_DIR)

# CDD DATA
CDD_DIR = 'intermediate_outputs_multi_model_energy'
base_temps = ['18p3', '20p0', '24p0', '26p0']
cdd_datasets = {}

for base_temp in base_temps:
    filepath = os.path.join(CDD_DIR, f'sdd_{base_temp}', f'ISIMIP3b_MM_sdd_c_{base_temp}.nc4')
    key = f'cdd_{base_temp}'
    cdd_datasets[key] = read_netcdf(filepath)

# POPULATION DENSITY DATA (inhabitants / km2)
POP_DIR = 'GriddedPop_30min'
ssps = ['SSP1', 'SSP2', 'SSP3', 'SSP5']
popd_datasets = {}

for ssp in ssps:
    filepath = os.path.join(POP_DIR, f'{ssp}', f'GPOPD_30MIN.NC')
    key = f'pop_{ssp}'
    popd_datasets[key] = read_netcdf(filepath)

# IMAGE REGIONS DATA
image_regions = read_netcdf(r'Gridded_IMAGE_data\GREG_30MIN.NC')

# AREA DATA (km2 / grid cell)
area = read_netcdf(r'Gridded_IMAGE_data\GAREACELLNOWATER_30MIN.NC')

# TEMPERATURE CHANGE TIME SERIES (TIMER OUTPUT)
gwl = pd.read_csv(r'GWL\GWL_SSP5_H.csv') 

**INTERPOLATE CDD DATA FOR ANY GLOBAL WARMING LEVEL**

In [5]:
def interpolate_cdd(dataset, target_warming_level, stats_value='mean'):
    """
    Interpolate CDD data for a specific warming level using the existing threshold values.
    
        Parameters:
    dataset : xarray.Dataset containing CDD data
    target_warming_level : float - desired warming level to interpolate to (between min and max threshold values)
    stats_value : str, default='mean'
        
        Returns:
    xarray.DataArray of interpolated CDD values for the target warming level
    """
    # Extract the threshold values and ensure they're sorted
    thresholds = dataset.threshold.values
    
    # Check input target warming level is within bounds
    if target_warming_level < thresholds.min() or target_warming_level > thresholds.max():
        raise ValueError(f"Target warming level {target_warming_level} is outside the available range "f"[{thresholds.min()}, {thresholds.max()}]")
    
    # select data and interpolate through GWLs
    cdd_data = dataset.sdd_c.sel(stats=stats_value)
    interpolated = cdd_data.interp(threshold=target_warming_level)
    
    # Set coordinates and attributes
    interpolated = interpolated.assign_coords({
        'warming_level': target_warming_level
    })
    
    # Add metadata
    interpolated.attrs['description'] = f'Interpolated CDD values for {target_warming_level}°C warming level'
    interpolated.attrs['stats_value'] = stats_value
    
    return interpolated

**INTERPOLATE CDD DATA FOR A TIME SERIES OF WARMING LEVELS**

In [6]:
def filter_gwl(dataset, gwl_timeseries):

    thresholds = dataset.threshold.values
    min_threshold = thresholds.min()
    max_threshold = thresholds.max()

    valid_gwl_timeseries = gwl_timeseries[
        (gwl_timeseries['gwl'] >= min_threshold) & 
        (gwl_timeseries['gwl'] <= max_threshold)
    ].reset_index(drop=True)
    
    invalid_count = len(gwl_timeseries) - len(valid_gwl_timeseries)
    if invalid_count > 0:
        print(f"Filtered out {invalid_count} years with GWL values outside range [{min_threshold}, {max_threshold}]")
    
    return valid_gwl_timeseries


def interpolate_cdd_timeseries(dataset, gwl_timeseries, stats_value='mean'):
    """
    Interpolate CDD data for a time series of warming levels.
    
        Parameters:
    dataset : xarray.Dataset containing CDD data
    gwl_timeseries : pandas.DataFrame containing GWL values per year
    stats_value : str, default='mean' as statistical measure to use
        
        Returns:
    xarray.Dataset with interpolated CDD values for all years
    """

    interpolated_data = []
    
    for _, row in gwl_timeseries.iterrows():
        try:
            result = interpolate_cdd(dataset, row['gwl'], stats_value)
            result = result.assign_coords({'year': row['year']})
            interpolated_data.append(result)
        except ValueError as e:
            print(f"Warning: Could not interpolate for year {row['year']} (GWL: {row['gwl']}): {str(e)}")
            continue
    
    if not interpolated_data:
        raise ValueError("No valid interpolated data produced")
    
    combined = xr.concat(interpolated_data, dim='year')
    combined = combined.assign_coords({
        'warming_level': ('year', [d.warming_level.item() for d in interpolated_data])
    })
    
    combined.attrs['description'] = 'Time series of interpolated CDD values'
    combined.attrs['stats_value'] = stats_value
    combined.attrs['year_range'] = f"{gwl_timeseries['year'].min()}-{gwl_timeseries['year'].max()}"
    
    return combined

In [7]:
# Gridded CDDs per year for different base temperatures, based on yearly GWL

interpolated_cdd_timeseries = {}

for base_temp, dataset in cdd_datasets.items():
    # base_temp = temp
    
    valid_gwl = filter_gwl(dataset, gwl)

    interpolated_cdd_timeseries[base_temp] = interpolate_cdd_timeseries(
        dataset, 
        valid_gwl, 
        stats_value='mean'
    )
    print(f"Completed time series interpolation for base temperature {base_temp}")

# Dataset for multiple warming levels (as coords) and base temps (as data variables)
combined_cdd_dataset_timeseries = xr.Dataset(
    {f'{temp}': data for temp, data in interpolated_cdd_timeseries.items()}
)

Filtered out 55 years with GWL values outside range [1.2, 3.5]
Completed time series interpolation for base temperature cdd_18p3
Filtered out 55 years with GWL values outside range [1.2, 3.5]
Completed time series interpolation for base temperature cdd_20p0
Filtered out 55 years with GWL values outside range [1.2, 3.5]
Completed time series interpolation for base temperature cdd_24p0
Filtered out 55 years with GWL values outside range [1.2, 3.5]
Completed time series interpolation for base temperature cdd_26p0


In [23]:
# # MAPS OF CDDs PER YEAR

# toplot1 = interpolated_cdd_timeseries['cdd_24p0'].sel(year=2026, method = 'nearest')
# toplot2 = interpolated_cdd_timeseries['cdd_24p0'].sel(year=2060, method = 'nearest')
# toplot3 = interpolated_cdd_timeseries['cdd_24p0'].sel(year=2100, method = 'nearest')

# proj = ccrs.PlateCarree() # define the projection (PlateCarree for a global map)

# # Set common color limits for all maps
# vmin = 0
# vmax = max(toplot1.max(), toplot2.max(), toplot3.max())

# fig = plt.figure(figsize=(19,4))

# ax1 = fig.add_subplot(1,3,1, projection=proj)
# toplot1.plot.imshow( transform=proj, cmap='viridis', vmin=vmin, vmax=vmax, add_colorbar=True)
# ax1.set_title('CDDs in 2026')

# ax2= fig.add_subplot(1,3,2, projection=proj)
# toplot2.plot.imshow( transform=proj, cmap='viridis', vmin=vmin, vmax=vmax, add_colorbar=True)
# ax2.set_title('CDDs in 2060')

# ax3 = fig.add_subplot(1,3,3, projection=proj)
# toplot3.plot.imshow( transform=proj, cmap='viridis', vmin=vmin, vmax=vmax, add_colorbar=True)
# ax3.set_title('CDDs in 2100')

# # Display the plots
# plt.tight_layout()
# plt.show()

**IDENTIFY AREAS OF HEAT STRESS BASED ON YEARLY CDD THRESHOLD**

In [10]:
def get_cdd_threshold_masks(data, thresholds=[50, 100, 200, 400]):

    """
    Create masks showing where yearly CDD exceeds specified thresholds.
    This represents areas of heat stress exposure.
    
        Parameters:
    data : xarray.DataArray of CDD data with dimensions (lat, lon) or (warming_level, lat, lon)
    thresholds : list of CDD thresholds (integers)
        
        Returns:
    xarray.Dataset containing boolean masks for each CDD threshold
    """
    # Initialize dictionary to store masks
    masks = {}
    
    # Create mask for each threshold
    for threshold in thresholds:
        mask_name = f'mask_{threshold}'
        masks[mask_name] = data >= threshold
    
    # Combine into dataset
    ds = xr.Dataset(masks)
    
    return ds

In [11]:
# Areas of heat stress per year, CDD threshold and base temperature
heatstress_masks = {}

for base_temp, data in combined_cdd_dataset_timeseries.items():
    heatstress_masks[base_temp] = get_cdd_threshold_masks(data)
    print(f"Created high heat masks for CDDs with base temperature {base_temp}")

Created high heat masks for CDDs with base temperature cdd_18p3
Created high heat masks for CDDs with base temperature cdd_20p0
Created high heat masks for CDDs with base temperature cdd_24p0
Created high heat masks for CDDs with base temperature cdd_26p0


In [22]:
# MAP OF HEAT STRESSED AREAS

# vmin = 0
# vmax = 1

# # maps to plot
# toplot1 = heatstress_masks['cdd_20p0']['mask_200'].sel(year=2030, method = 'nearest')
# toplot2 = heatstress_masks['cdd_20p0']['mask_400'].sel(year=2030, method = 'nearest')

# toplot3 = heatstress_masks['cdd_24p0']['mask_200'].sel(year=2030, method = 'nearest')
# toplot4 = heatstress_masks['cdd_24p0']['mask_400'].sel(year=2030, method = 'nearest')

# # set up the plot
# proj = ccrs.PlateCarree() # define the projection (PlateCarree for a global map)
# fig = plt.figure(figsize=(11,5))

# ax1 = fig.add_subplot(2,2,1, projection=proj)
# toplot1.plot.imshow(ax=ax1, transform=proj, cmap='viridis', vmin=vmin, vmax=vmax, add_colorbar=True)
# ax1.set_title(f'Areas with >200 CCDs in 2030, with base temp 20dC')

# ax2= fig.add_subplot(2,2,2, projection=proj)
# toplot2.plot.imshow(ax=ax2, transform=proj, cmap='viridis', vmin=vmin, vmax=vmax, add_colorbar=True)
# ax2.set_title('Areas with >400 CCDs/year in 2030, with base temp 20dC')

# ax3 = fig.add_subplot(2,2,3, projection=proj)
# toplot3.plot.imshow(ax=ax3, transform=proj, cmap='viridis', vmin=vmin, vmax=vmax, add_colorbar=True)
# ax3.set_title('Areas with >200 CCDs/year in 2030, with base temp 24dC')

# ax4 = fig.add_subplot(2,2,4, projection=proj)
# toplot4.plot.imshow(ax=ax4, transform=proj, cmap='viridis', vmin=vmin, vmax=vmax, add_colorbar=True)
# ax4.set_title('Areas with >400 CCDs/year in 2030, with base temp 24dC')

# # Display the plots
# plt.tight_layout()
# plt.show()

In [12]:
def calculate_population(gpopd_dataset, area_dataset, year):
    """
    Calculate population per grid cell for a specific year.
    
        Parameters:
    gpopd_dataset : xarray.Dataset of population density (persons per km²)
    area_dataset : xarray.Dataset of grid cell areas in km²
    year : int
    
        Returns:
    xarray.DataArray of population per grid cell
    """
    # Select the specified year for both datasets
    time = pd.Timestamp(f"{year}-01-01")
    pop_density = gpopd_dataset.GPOPD_30MIN.sel(time=time, method='nearest')
    area = area_dataset.GAREACELLNOWATER_30MIN.sel(time=time, method='nearest')
    
    # Calculate population by multiplying density by area
    population = pop_density * area
    
    return population

In [86]:
# Population data (pop / grid cell) per year and SSP
pop_datasets = {}

for ssp, dataset in popd_datasets.items():

    yearly_population = []

    # Extract years from the time dimension
    years = pd.DatetimeIndex(dataset['time'].values).year.unique()

    for year in years:
        pop_data = calculate_population(dataset, area, year)
        yearly_population.append(pop_data.expand_dims(year=[year]))

    # Combine yearly data into a single dataset for each SSP
    pop_datasets[ssp] = xr.concat(yearly_population, dim="year")
    print(f'added pop data for {ssp} to pop datasets')

added pop data for pop_SSP1 to pop datasets
added pop data for pop_SSP2 to pop datasets
added pop data for pop_SSP3 to pop datasets
added pop data for pop_SSP5 to pop datasets


In [89]:
def interpolate_population_timeseries(pop_datasets, interpolation_year=2026):
    """
    Interpolate population data between specified years for multiple SSP scenarios
    
        Parameters:
    pop_datasets : dict of xarray.DataArrays representing population data for different SSPs
    interpolation_years : integer year between which to interpolate (default: 2026)
    
        Returns:
    xarray.Dataset with interpolated population data for specified year
    """

    interpolated_pop_data = {}
    
    for ssp, dataset in pop_datasets.items():
        try:
            dataset = dataset.assign_coords(
                    year=dataset.year.values)

            interpolated = dataset.interp(
                year=interpolation_year
            )
            
            interpolated_pop_data[ssp] = interpolated
        except Exception as e:
            print(f"Warning: Could not interpolate population for {ssp}: {str(e)}")
            continue

    combined = xr.Dataset(interpolated_pop_data)
    
    return combined

interpolated_pop_2026 = interpolate_population_timeseries(pop_datasets)

In [90]:
def add_interpolated_year_to_datasets(pop_datasets, interpolated_pop_dataset, interpolation_year=2026):
    """
    Add interpolated population data for a specific year to existing population datasets.
    
        Parameters:
    pop_datasets : dict of xarray.DataArrays representing population data
    interpolated_pop_dataset : xarray.Dataset with interpolated population data for the specified year
    interpolation_year : integer of year being added (default: 2026)
    
        Returns:
    dict of updated population datasets with the new year inserted in the correct position
    """
    updated_pop_datasets = {}
    
    for ssp in interpolated_pop_dataset.data_vars:
        original_dataset = pop_datasets[ssp]
        interpolated_data = interpolated_pop_dataset[ssp]
        
        existing_years = original_dataset.year.values
        insert_index = np.searchsorted(existing_years, interpolation_year)
        
        # Create a new array of years and data with the interpolated year and data inserted
        new_years = np.insert(existing_years, insert_index, interpolation_year)
        new_data = np.insert(
            original_dataset.values, 
            insert_index, 
            interpolated_data.values, 
            axis=0
        )
        
        # Recreate dataset with the new years and data
        updated_dataset = xr.DataArray(
            new_data, 
            coords={
                'year': new_years, 
                'latitude': original_dataset.latitude, 
                'longitude': original_dataset.longitude
            }, 
            dims=['year', 'latitude', 'longitude']
        )

        updated_pop_datasets[ssp] = updated_dataset
    
    return updated_pop_datasets

pop_datasets = add_interpolated_year_to_datasets(pop_datasets, interpolated_pop_2026)

In [93]:
def calculate_regional_population(population_data, region_data):
    """
    Calculate total population for each region.
    
        Parameters:
    population_data : xarray.DataArray of population per grid cell
    region_data : xarray.Dataset of region classifications
    
        Returns :
    A dictionary with region numbers as keys and total population as values
    """

    # Convert to numpy arrays for faster computation
    pop_array = population_data.values
    reg_array = region_data.GREG_30MIN.isel(time=0).values
    
    # Initialize results dictionary
    regional_populations = {}
    
    # Calculate for each region
    for region_num in range(1, 27):
        mask = (reg_array == region_num)
        regional_pop = np.sum(pop_array[mask])
        regional_populations[region_num] = regional_pop
        
    return regional_populations

In [94]:
def get_regional_population(pop_dataset, region_dataset, year):
    """
    Calculate regional populations for a specific year.
    
        Parameters:
    pop_dataset : xarray.Dataset of population per grid cell
    region_dataset : xarray.Dataset of region classifications (1-27)
    year : int
        
        Returns:
    dict with region numbers as keys and population values
    """
    
    # Select the specific year's data
    pop_data_year = pop_dataset.sel(year=year)

    # Calculate population per region
    regional_pops = calculate_regional_population(pop_data_year, region_dataset)
    
    # Print results
    # print(f"\nRegional Populations for {year} :")
    # print("-" * 40)
    # total_pop = 0
    # for region, pop in regional_pops.items():
    #     pop_millions = pop / 1e6
    #     print(f"Region {region:1d}: {pop_millions:,.2f} million")
    #     total_pop += pop

    return regional_pops

In [135]:
# Calculate regional population per SSP and year

SSPs = [key for key in pop_datasets.keys()]

pop_data_dic = []

# Iterate over all combinations
for ssp in SSPs:
    years = pop_datasets[ssp].coords['year'].to_numpy()    

    for year in years:
        try:
            reg_pop = get_regional_population(
                pop_datasets[ssp],
                image_regions,
                year
            )

            for key, value in reg_pop.items():
                pop_data_dic.append({
                    'SSP': ssp,
                    'Year': year,
                    'Region': key,
                    'RegionalPop': value
                })
        except Exception as e:
            print(f"Error adding data to pop_data_dic {ssp}, {year}: {e}")
    
    print(f"Processed pop data successfully for {ssp}")

regional_pop = pd.DataFrame(pop_data_dic)

Processed pop data successfully for pop_SSP1
Processed pop data successfully for pop_SSP2
Processed pop data successfully for pop_SSP3
Processed pop data successfully for pop_SSP5


**ESTIMATE POPULATION EXPOSED TO HEAT STRESS**

In [138]:
def get_heatstress_pop(pop_data, region_data, heatstress_dataset, year, SSP, base_temp, mask_threshold):
    """
    Calculate total population for each region in areas of heat stress
        
        Parameters:
    pop_data: xarray.DataArray of pop per grid cell
    region_data: xarray.DataSet with region classification
    heatstress_dataset: xarray.DataArray - boolean mask indicating areas of heat stress (True)
    year: int
    SSP: string
    base_temp: string
    mask_threshold: string

        Returns
    Dictionary with regions as keys and heat-stress pop as values
    """
 
    # get heat stress mask for the specified year, mask and rename coordinates
    mask = heatstress_dataset[mask_threshold].sel(year=year).rename({'lon': 'longitude', 'lat': 'latitude'})

    # select data based on year and SSP and convert into numpy arrays
    pop_array = pop_data[SSP].sel(year=year).values
    reg_array = region_data.GREG_30MIN.isel(time=0).values
    mask_array = mask.values

    # initialise results dict
    regional_heatstress_pop = {}
    regional_heatstress_pop_share = {}

    ### input validation ###
    if not np.any(mask_array):
        print(f"Warning: Heat stress mask is empty for {year}, {base_temp}, {mask_threshold}")

    if np.all(np.isnan(pop_array)):
        print(f"Warning: Population data contains only NaN values for {year}, {SSP}")

    for region in range(1,27):

        # create combined mask for region and heat stress
        region_mask = (reg_array == region)
        combined_mask = region_mask & mask_array

        # Debugging
        total_region_pop = np.nansum(pop_array[region_mask])
        # print(f"Total pop in region {region}: {total_region_pop:,.2f}")

        # # calculate population pop in high heat areas for this region
        exposed_pop = np.nansum(pop_array[combined_mask])
        regional_heatstress_pop[region] = exposed_pop
        # print(f"Pop in high heat areas in region {region} at {base_temp} and {mask_threshold} in {year}: {exposed_pop:,.2f}")

        # # calculate % of exposed population to high heat for this region
        #regional_heatstress_pop_share = exposed_pop / total_region_pop * 100
        regional_heatstress_pop_share[region] = exposed_pop / total_region_pop * 100
        # if exposed_pop > 0:
        #     print(f"% of region {region} pop exposed: {(regional_heatstress_pop_share):,.2f}%")

    # return regional_heatstress_pop
    return regional_heatstress_pop_share

In [140]:
# 1. Extract common years between population and heat stress data
common_years = sorted(set(pop_datasets[next(iter(pop_datasets))].year.values).intersection(
    *[heatstress_masks[base_temp].year.values for base_temp in heatstress_masks]
))

# 2. Extract SSPs, base temps and Mask Thresholds
SSPs = list(pop_datasets.keys())
base_temps = list(heatstress_masks.keys())
mask_thresholds = [var for var in heatstress_masks[next(iter(base_temps))].data_vars 
                  if var.startswith('mask_')]

# 3. Initialize results container
pop_exposed_data = []

# 4. Iterate over all combinations
for base_temp in base_temps:
    heatstress_dataset = heatstress_masks[base_temp]
    if base_temp not in heatstress_masks:
        print(f"Error: base_temp '{base_temp}' is not in heatstress_masks")
        continue

    for mask_threshold in mask_thresholds:
        for ssp in SSPs:
            for year in common_years:
                try:
                    heatstress_pop = get_heatstress_pop(
                        pop_data=pop_datasets,
                        region_data=image_regions,
                        heatstress_dataset=heatstress_dataset,
                        year=year,
                        SSP=ssp,
                        base_temp=base_temp,
                        mask_threshold=mask_threshold
                    )

                    # Add all regions to results
                    pop_exposed_data.extend([
                        {
                            'BaseTemp': base_temp,
                            'MaskThreshold': mask_threshold,
                            'SSP': ssp,
                            'Year': year,
                            'Region': region,
                            'ExposedPopulationShare': exposed_pop
                        } for region, exposed_pop in heatstress_pop.items()
                    ])
                    
                    # print(f"Processed: {base_temp}, {mask_threshold}, {ssp}, {year}")
                    
                except Exception as e:
                    print(f"Error processing {base_temp}, {mask_threshold}, {ssp}, {year}: {e}")

exposed_pop = pd.DataFrame(pop_exposed_data)

**CALCULATE REGIONAL POPULATION-WEIGHTED CDDS**

In [144]:
def calculate_population_weighted_cdds(pop_dataset, cdd_dataset, region_dataset, year, base_temp):
    """
    Calculate population-weighted Cooling Degree Days (CDDs) for each region.
    
        Parameters:
    pop_dataset : xarray.Dataset of population per grid cell data
    cdd_dataset : xarray.Dataset of cooling degree days data
    region_dataset : xarray.Dataset of iamge region classifications
    base_temp : str, optional, CDD variable to use (default='cdd_18p3')
    year : int

        Returns:
    dict with region numbers as keys and population-weighted CDDs as values
    """
    
    # Extract data from datasets
    regions = region_dataset.GREG_30MIN.isel(time=0)
    cdd_selected = cdd_dataset[base_temp].sel(year=year)
    
    # Align regions with population coordinates by rounding coordinates to the 5th decimal
    regions = regions.assign_coords(
        latitude=np.round(regions.latitude, 5),
        longitude=np.round(regions.longitude, 5)
    )
    population = pop_dataset.assign_coords(
        latitude=np.round(pop_dataset.latitude, 5),
        longitude=np.round(pop_dataset.longitude, 5)
    )

    # Regrid CDD data to match population grid
    cdd_regridded = cdd_selected.interp(
        lat=population.latitude,
        lon=population.longitude
    )

    # Initialize dictionary for results
    regional_weighted_cdds = {}
    
    # Get unique region numbers
    unique_regions = np.unique(regions.values[~np.isnan(regions.values)])
    
    # Calculate population-weighted CDDs for each region
    for region in unique_regions:
        # Create mask for current region
        region_mask = (regions == region)
        
        # Extract regional population and CDDs
        region_pop = population.where(region_mask)
        region_cdds = cdd_regridded.where(region_mask)
        
        # Calculate total population in region
        total_pop = region_pop.sum().values
        
        if total_pop > 0:  # Avoid division by zero
            # Calculate population-weighted average CDDs
            weighted_cdds = (
                (region_pop * region_cdds).sum().values / total_pop
            )
            regional_weighted_cdds[int(region)] = float(weighted_cdds)
        else:
            regional_weighted_cdds[int(region)] = 0.0
    
    return regional_weighted_cdds

In [145]:
# Define the years to process
years = [2026, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100]

# Initialize results container
combined_cdd_gwl_data = []

# Iterate over all combinations
for base_temp in base_temps:
    for ssp in SSPs:
        for year in years:
            try:
                weighted_cdds = calculate_population_weighted_cdds(
                    pop_dataset=pop_datasets[ssp],
                    cdd_dataset=combined_cdd_dataset_timeseries,
                    region_dataset=image_regions,
                    year=year,
                    base_temp = base_temp
                )
                
                # All results for each region
                combined_cdd_gwl_data.extend([
                    {
                        'SSP': ssp,
                        'Year': year,
                        'BaseTemp': base_temp,
                        'Region': region,
                        'weighted_cdd': cdd_value,
                        'warming_level': float(combined_cdd_dataset_timeseries.warming_level.sel(year=year).values)
                    } for region, cdd_value in weighted_cdds.items()
                ])

                print(f"Processed successfully for {base_temp}, {ssp}, {year}")

            except Exception as e:
                print(f"Error processing {base_temp}, {ssp}, {year}: {str(e)}")

pop_weighted_cdd = pd.DataFrame(combined_cdd_gwl_data)

Processed successfully for cdd_18p3, pop_SSP1, 2026
Processed successfully for cdd_18p3, pop_SSP1, 2030
Processed successfully for cdd_18p3, pop_SSP1, 2040
Processed successfully for cdd_18p3, pop_SSP1, 2050
Processed successfully for cdd_18p3, pop_SSP1, 2060
Processed successfully for cdd_18p3, pop_SSP1, 2070
Processed successfully for cdd_18p3, pop_SSP1, 2080
Processed successfully for cdd_18p3, pop_SSP1, 2090
Processed successfully for cdd_18p3, pop_SSP1, 2100
Processed successfully for cdd_18p3, pop_SSP2, 2026
Processed successfully for cdd_18p3, pop_SSP2, 2030
Processed successfully for cdd_18p3, pop_SSP2, 2040
Processed successfully for cdd_18p3, pop_SSP2, 2050
Processed successfully for cdd_18p3, pop_SSP2, 2060
Processed successfully for cdd_18p3, pop_SSP2, 2070
Processed successfully for cdd_18p3, pop_SSP2, 2080
Processed successfully for cdd_18p3, pop_SSP2, 2090
Processed successfully for cdd_18p3, pop_SSP2, 2100
Processed successfully for cdd_18p3, pop_SSP3, 2026
Processed su

**MERGE HEATSTRESS ESTIMATE WITH REGIONAL CDDs**

In [146]:
# Extract common years between exposed population and cdd data and filter
cdd_years = pop_weighted_cdd['Year'].unique()
exposed_years = exposed_pop['Year'].unique()
common_years = np.intersect1d(cdd_years, exposed_years)
# print(common_years) # check if same as other common_years. if so delete

columns = ["BaseTemp", "Region", "SSP", "MaskThreshold", "warming_level"]
# merge regional cdd data with exposed_pop
exposed_pop_cdd = (exposed_pop[exposed_pop['Year'].isin(common_years)]
                   .merge(
                       pop_weighted_cdd[pop_weighted_cdd['Year'].isin(common_years)]
                       [['BaseTemp', 'SSP', 'Year', 'Region', 'warming_level']],
                        on=['BaseTemp', 'SSP', 'Year', 'Region'],
                        how='left'
                    )
                    .merge(
                        regional_pop,
                        on=['SSP', 'Year', 'Region']
                    ).set_index(columns)
                    .dropna())


exposed_pop_series = exposed_pop_cdd['ExposedPopulationShare']

**Create mym file for TIMER**

In [147]:
def write_mym(df, output_dir, value_column='ExposedPopulationShare', year_column='Year', base_temp='cdd_24p0'):
    
    os.makedirs(output_dir, exist_ok=True)

    ssp_list = df.index.get_level_values("SSP").unique()
    mask_order = df.index.get_level_values("MaskThreshold").unique().tolist()
    warming_order = df.index.get_level_values("warming_level").unique().tolist()
    
    for ssp in ssp_list:
        try:
            data_array = (
                df
                .xs(
                    (ssp, base_temp),
                    level=["SSP", "BaseTemp"],
                )
                .to_xarray()
            )

            # Convert to xarray using from_dataframe directly
            data_array = data_array.reindex({"Region": range(1, 27)})
            
            # Convert DataArray to series and unstack
            df_pym = data_array.to_series()
            
            # Reorder levels but don't sort - this preserves the original order
            df_pym = df_pym.reorder_levels(["Region", "MaskThreshold", "warming_level"])
            
            regions = range(1, 27)
            new_index = pd.MultiIndex.from_product([regions, mask_order, warming_order], names=["Region", "MaskThreshold", "warming_level"])
            df_pym = df_pym.reindex(new_index)
            print(df_pym)

            # Save file
            ssp_id = ssp[4:9]
            filename = f'heat_exposed_pop_share_{ssp_id}.dat'
            filepath = os.path.join(output_dir, filename)
            pym.write_mym(data=df_pym, filename=filepath, variable_name=value_column)
            print(f"Saved {filepath}")
            
        except Exception as e:
            print(f"Error processing {ssp}: {str(e)}")
            import traceback
            traceback.print_exc()
            
    return df_pym

output_dir = os.path.join(ROOT_DIR, 'EXPOSED_POP_FOR_TIMER_NEW')
final = write_mym(exposed_pop_series, output_dir)

Region  MaskThreshold  warming_level
1       mask_50        1.214875          1.519929
                       1.311853          1.905137
                       1.573306          5.761435
                       1.867270         17.129506
                       2.175638         35.552028
                                          ...    
26      mask_400       2.175638         49.633083
                       2.480449         58.093637
                       2.783638         66.296947
                       3.083198         71.420813
                       3.373593         74.966609
Name: ExposedPopulationShare, Length: 936, dtype: float64
Saved X:\user\crassierc\code\cooling\CoolingGap\EXPOSED_POP_FOR_TIMER_NEW\heat_exposed_pop_share_SSP1.dat
Region  MaskThreshold  warming_level
1       mask_50        1.214875          1.519998
                       1.311853          1.905525
                       1.573306          5.764406
                       1.867270         17.137849
            

In [25]:
### this is to have gwl levels instead of years
# def write_mym(df, output_dir, value_column='ExposedPopulationShare', year_column='Year', base_temp='cdd_24p0'):
    
#     os.makedirs(output_dir, exist_ok=True)

#     ssp_list = df.index.get_level_values("SSP").unique()
#     mask_order = df.index.get_level_values("MaskThreshold").unique().tolist()
    
#     for ssp in ssp_list:
#         try:
#             data_array = (
#                 df
#                 .xs(
#                     (ssp, base_temp),
#                     level=["SSP", "BaseTemp"],
#                 )
#                 .to_xarray()
#             )

#             # Convert to xarray using from_dataframe directly
#             data_array = data_array.reindex({"Region": range(1, 27)})
            
#             # Convert DataArray to series and unstack
#             df_pym = data_array.to_series().unstack("warming_level")
            
#             # Reorder levels but don't sort - this preserves the original order
#             df_pym = df_pym.reorder_levels(["Region", "MaskThreshold"])
            
#             regions = range(1, 27)
#             new_index = pd.MultiIndex.from_product([regions, mask_order], names=["Region", "MaskThreshold"])
#             df_pym = df_pym.reindex(new_index)

#             # Save file
#             filename = f'exposed_pop_heat_share_{ssp}_new.dat'
#             filepath = os.path.join(output_dir, filename)
#             pym.write_mym(data=df_pym, filename=filepath, variable_name=value_column)
#             print(f"Saved {filepath}")
            
#         except Exception as e:
#             print(f"Error processing {ssp}: {str(e)}")
#             import traceback
#             traceback.print_exc()
            
#     return df_pym

output_dir = os.path.join(ROOT_DIR, 'EXPOSED_POP_FOR_TIMER')
final = write_mym(exposed_pop_series, output_dir)

Saved X:\user\crassierc\code\cooling\CoolingDegreeDays\EXPOSED_POP_FOR_TIMER\exposed_pop_heat_share_pop_SSP1_new.dat
Saved X:\user\crassierc\code\cooling\CoolingDegreeDays\EXPOSED_POP_FOR_TIMER\exposed_pop_heat_share_pop_SSP2_new.dat
Saved X:\user\crassierc\code\cooling\CoolingDegreeDays\EXPOSED_POP_FOR_TIMER\exposed_pop_heat_share_pop_SSP3_new.dat
Saved X:\user\crassierc\code\cooling\CoolingDegreeDays\EXPOSED_POP_FOR_TIMER\exposed_pop_heat_share_pop_SSP5_new.dat
